# TurboQuant VRAM Test

Run on Colab with a GPU runtime (A100 recommended for long context).
Compares baseline vs TurboQuant VRAM usage with the fused Triton attention kernel.

**Important:** TurboQuant saves KV cache memory. At short context (2K tokens),
the KV cache is only ~130 MB — invisible next to 16 GB of model weights.
You need **8K+ tokens** to see a meaningful VRAM difference. This notebook
defaults to 8K tokens where TurboQuant saves ~700+ MB of peak VRAM.

In [ ]:
!pip install -q turboquant transformers accelerate scipy

In [ ]:
import torch
import gc
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
try:
    import triton
    print(f"Triton version: {triton.__version__}")
except ImportError:
    print("Triton not installed — will use PyTorch fallback attention")

In [ ]:
from turboquant import TurboQuantSession

MODEL = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"

# 8K tokens — long enough that KV cache is a significant fraction of VRAM.
# At 2K tokens the KV cache is only ~130 MB (0.4% of total VRAM) — invisible.
# At 8K tokens the KV cache is ~1.1 GB — TurboQuant saves ~700-800 MB.
PROMPT = "The quick brown fox " * 2000  # ~8K tokens
BITS = 4
MAX_NEW = 32
print(f"Using dtype={DTYPE}")

In [ ]:
# --- Baseline ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

session_bl = TurboQuantSession.from_pretrained(
    MODEL, variant="baseline", bits=BITS,
    dtype=DTYPE, device_map="auto",
)
model_bytes_bl = torch.cuda.memory_allocated()
print(f"Model loaded: {model_bytes_bl / 1e9:.2f} GB")

text_bl = session_bl.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)
peak_bl = torch.cuda.max_memory_allocated()
telem_bl = session_bl.last_telemetry()

print(f"Baseline peak VRAM: {peak_bl / 1e9:.2f} GB")
print(f"Baseline gen time: {telem_bl.get('generation_seconds', 'N/A')}s")
print(f"Output: {text_bl[:200]}...")

del session_bl
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# --- TurboQuant ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

session_tq = TurboQuantSession.from_pretrained(
    MODEL, variant="qmse_packed", bits=BITS,
    dtype=DTYPE, device_map="auto",
)
model_bytes_tq = torch.cuda.memory_allocated()
print(f"Model loaded: {model_bytes_tq / 1e9:.2f} GB")

text_tq = session_tq.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)
peak_tq = torch.cuda.max_memory_allocated()
telem_tq = session_tq.last_telemetry()

print(f"TurboQuant peak VRAM: {peak_tq / 1e9:.2f} GB")
print(f"TurboQuant gen time: {telem_tq.get('generation_seconds', 'N/A')}s")
print(f"Packed KV: {telem_tq.get('packed_actual_bytes', 0) / 1e6:.0f} MB")
print(f"Dense KV would be: {telem_tq.get('dense_kv_bytes', 0) / 1e6:.0f} MB")
print(f"Payload savings: {telem_tq.get('payload_savings_percent', 0):.1f}%")
print(f"Output: {text_tq[:200]}...")

In [ ]:
# --- Comparison ---
peak_saved_gb = (peak_bl - peak_tq) / 1e9
dense_kv_mb = telem_tq.get('dense_kv_bytes', 0) / 1e6
packed_kv_mb = telem_tq.get('packed_actual_bytes', 0) / 1e6
payload_pct = telem_tq.get('payload_savings_percent', 0)

print("=" * 60)
print(f"{'Metric':<30} {'Baseline':>14} {'TurboQuant':>14}")
print("-" * 60)
print(f"{'Model loaded (GB)':<30} {model_bytes_bl/1e9:>14.2f} {model_bytes_tq/1e9:>14.2f}")
print(f"{'Peak VRAM (GB)':<30} {peak_bl/1e9:>14.2f} {peak_tq/1e9:>14.2f}")
print(f"{'Peak VRAM saved (GB)':<30} {'':>14} {peak_saved_gb:>14.2f}")
print(f"{'Dense KV size (MB)':<30} {dense_kv_mb:>14.0f} {dense_kv_mb:>14.0f}")
print(f"{'Packed KV size (MB)':<30} {'N/A':>14} {packed_kv_mb:>14.0f}")
print(f"{'KV payload savings':<30} {'':>14} {payload_pct:>13.1f}%")
print(f"{'Gen time (s)':<30} {telem_bl.get('generation_seconds','?'):>14} {telem_tq.get('generation_seconds','?'):>14}")
print(f"{'Output match':<30} {'':>14} {str(text_bl[:100] == text_tq[:100]):>14}")
print("=" * 60)

if model_bytes_bl / 1e9 > 25:
    print()
    print("WARNING: Model loaded as >25 GB. It may be in FP32 instead of BF16.")
    print(f"Check: DTYPE={DTYPE}. If FP32, peak VRAM is inflated by ~2x.")